---
# Hyperparameter Tuning — Grid Search on v4.0 Models

Runs `RandomizedSearchCV` over LR, RF, and XGB using the same data splits
as the v4.0 training run (no GCS reload). Saves tuned params as a new
hyperparam version and snapshots model artifacts + validation results.

In [16]:
import sys
import os
from pprint import pprint

# Adds 'src/capstone' to the path so 'pipeline' is discoverable
# assuming the notebook is in ROOT/notebooks/ and code is in ROOT/src/capstone/
sys.path.append(os.path.abspath("../"))

import pandas as pd

from pipeline.version_config import VersionConfig
from pipeline.pipeline_run import PipelineRun
from pipeline.factory import PipelineFactory
from utils.tune_hyperparameters import get_default_param_grid_by_name, get_all_default_param_grids

In [17]:
# Set False only when you're ready to write to GCS.
dry_run = False

# Set HyperParam Grid parameters

Grid defaults exist in `utils/tune_hyperparameters.py`

In [18]:
"""
_PARAM_GRIDS = {
    "LogisticRegression": {
        "C":        [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 100.0],
        "penalty":  ["l1"],        # keep L1 — established in project
        "solver":   ["saga"],      # only solver that supports L1 + large datasets
        "max_iter": [2000, 5000],
    },
    "RandomForestClassifier": {
        "n_estimators":     [100, 200, 300, 500],
        "max_depth":        [None, 5, 10, 20, 30],
        "min_samples_leaf": [1, 2, 5, 10],
        "min_samples_split":[2, 5, 10],
        "max_features":     ["sqrt", "log2", None],
    },
    "XGBClassifier": {
        "n_estimators":      [100, 200, 300, 500],
        "max_depth":         [3, 4, 5, 6, 8],
        "learning_rate":     [0.01, 0.05, 0.1, 0.2],
        "subsample":         [0.6, 0.7, 0.8, 1.0],
        "colsample_bytree":  [0.6, 0.7, 0.8, 1.0],
        "min_child_weight":  [1, 3, 5, 10],
        "gamma":             [0, 0.1, 0.3, 0.5],
    },
}
"""
# Start with existing grid, then override
new_grids = get_all_default_param_grids()
pprint(new_grids)

{'LogisticRegression': {'C': [0.1, 1.0, 10.0],
                        'l1_ratio': [0.1, 0.5, 0.9],
                        'max_iter': [2000, 5000],
                        'penalty': ['elasticnet'],
                        'solver': ['saga']},
 'RandomForestClassifier': {'max_depth': [None, 5, 10, 20, 30],
                            'max_features': ['sqrt', 'log2'],
                            'min_samples_leaf': [2, 5, 10],
                            'min_samples_split': [2, 5, 10],
                            'n_estimators': [300, 500, 800]},
 'XGBClassifier': {'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
                   'gamma': [0, 0.1, 0.3, 0.5],
                   'learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2],
                   'max_bin': [256],
                   'max_depth': [3, 4, 5, 6, 8],
                   'min_child_weight': [1, 3, 5, 10],
                   'n_estimators': [300, 500, 1000, 1500],
                   'reg_alpha': [0, 0.01, 0.1, 1],
                   

In [19]:
new_grids["XGBClassifier"] = {
    "colsample_bytree": [0.6, 0.7, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3, 0.5],
    "max_depth": [3, 4, 5, 6, 8],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.6, 0.7, 0.8, 1.0],
    # Modified / added:
    "learning_rate": [0.005, 0.01, 0.05, 0.1, 0.2],
    "n_estimators": [300, 500, 1000, 1500],
    "tree_method": ["hist"],
    "max_bin": [
        256
    ],  # note: 256 is default; incrase for more accuracy at higher cost
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 1.5, 2, 5],
}

new_grids["RandomForestClassifier"] = {
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    # Modified / added:
    "max_features": ["sqrt", "log2"],
    "min_samples_leaf": [2, 5, 10],
    "n_estimators": [300, 500, 800],
}

new_grids["LogisticRegression"] = {
    "max_iter": [2000, 5000],
    "solver": ["saga"],
    # Modified / added:
    "penalty": ["elasticnet"],
    "l1_ratio": [0.1, 0.5, 0.9],  # For ElasticNet
    "C": [0.1, 1.0, 10.0],  # Narrowed down
}

pprint(new_grids)

{'LogisticRegression': {'C': [0.1, 1.0, 10.0],
                        'l1_ratio': [0.1, 0.5, 0.9],
                        'max_iter': [2000, 5000],
                        'penalty': ['elasticnet'],
                        'solver': ['saga']},
 'RandomForestClassifier': {'max_depth': [None, 5, 10, 20, 30],
                            'max_features': ['sqrt', 'log2'],
                            'min_samples_leaf': [2, 5, 10],
                            'min_samples_split': [2, 5, 10],
                            'n_estimators': [300, 500, 800]},
 'XGBClassifier': {'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
                   'gamma': [0, 0.1, 0.3, 0.5],
                   'learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2],
                   'max_bin': [256],
                   'max_depth': [3, 4, 5, 6, 8],
                   'min_child_weight': [1, 3, 5, 10],
                   'n_estimators': [300, 500, 1000, 1500],
                   'reg_alpha': [0, 0.01, 0.1, 1],
                   

## Tuning Config

In [20]:
# Add new grids here


config_tune = (
    VersionConfig.load(use_synthetic=False)
    .tune(strategy="random", n_iter=100, cv=5, new_grids=new_grids)
    .snapshot_hyperparams()
    .snapshot_models()
    .build()
)
run_tune = PipelineRun(config_tune)
stages_tune = PipelineFactory.tune_hyperparams(config_tune)

print('\nScenario         :', stages_tune.scenario)
print('\nStages: ', stages_tune)
print('current model_version    :', config_tune.model_version)
print('current hyperparam_version:', config_tune.hyperparam_version)

VersionConfig loaded:
  data:          v3.4 (raw_suffix='real')
  baselines:     v4.0
  model:         v5.0
  hyperparams:   v1.0
  use_synthetic: False

VersionConfig ready:
  Active flags      : ['models', 'hyperparams', 'tune']
  data              : v3.4 (unchanged)
  baselines         : v4.0 (unchanged)
  model             : v5.0 -> v5.1
  hyperparams       : v1.0 -> v1.1
  raw_version       : v3.4_real
  next_final_version: v3.4_100real
  model_version     : v5.0  ->  v5.1
  baselines_version : v4.0
  hyperparam_version: v1.0  ->  v1.1
  use_synthetic     : False
  Tuning            : strategy=random, n_iter=100, cv=5, scoring=roc_auc

  Call config.commit() after all snapshots succeed.

Scenario         : tune_hyperparams

Stages:  PipelineStages(scenario='tune_hyperparams', present=['loader', 'preprocessor', 'raw_snapshotter', 'engineer', 'splitter', 'scaler', 'augmenter', 'final_snapshotter', 'trainer', 'hyperparam_snapshotter', 'model_snapshotter', 'model_loader', 'validator',

# Load, process, engineer, split and scale the data

In [21]:
# Load is expensive, so splitting out
stages_tune.loader.run(run_tune)
run_tune.summary()

Loaded snapshot 'v3.4_real': 60696 rows from 2026-04-29
  Polls: {'upload': 21398, '24h': 20906, '7d': 18392}
Loaded baselines 'v4.0': 28814 baseline videos, 974 median rows (974 channels)
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 1), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models', 'hyperparams', 'tune']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       None
  df_engineered  None
  df_train       None
  df_test        None
  df_val         None
  X_train        None
  X_test         None
  X_val          None
  X_val_unscaled  None
  y_train        None
  y_test         None
  y_val          None
  models         empty dict
  results        empty dict


In [22]:
stages_tune.preprocessor.run(run_tune)
stages_tune.engineer.run(run_tune)
stages_tune.splitter.run(run_tune)
stages_tune.scaler.run(run_tune)
run_tune.summary()

Building clean dataset
snapshot cols: Index(['video_id', 'poll_timestamp', 'channel_id', 'channel_handle', 'title',
       'view_count', 'like_count', 'comment_count', 'face_count', 'brightness',
       'colorfulness', 'vertical', 'tier', 'description', 'tags',
       'duration_seconds', 'category_id', 'category_name', 'published_at',
       'poll_label', 'hours_since_publish', 'subscriber_count',
       'contains_synthetic_media'],
      dtype='object')

[1/3] Pivoting snapshots...
  Videos with all 3 polls: 18334 (dropped 3111 incomplete)
  Pivoted shape: (18403, 34)

[2/3] Joining baseline medians...
  Baseline join: 18403/18403 videos matched a channel median

[3/3] Cleaning data...
  Cleaned: 18403 rows × 40 columns

Clean dataset: 18403 rows × 40 columns
  all: dropped 414 rows with NaN in a baseline_median_* column or 0.0 baseline_median_engagement_rate
Engineering features

[1/9] Computing target variable...
  Target distribution: 55.1% above baseline, 44.9% below

[2/9] Comput

## Verify v4.0 Data Splits

Verify data before moving ahead to `ModelTrainer`

In [23]:
print(f"X_train len: {len(run_tune.X_train)}")
print(stages_tune.scaler.scaler_)
pprint(stages_tune.scaler.feature_cols_)

X_train len: 10236
StandardScaler()
['duration_seconds',
 'brightness',
 'colorfulness',
 'view_count_upload',
 'like_count_upload',
 'comment_count_upload',
 'subscriber_count_upload',
 'view_count_24h',
 'like_count_24h',
 'comment_count_24h',
 'subscriber_count_24h',
 'baseline_baseline_video_count',
 'view_count_velocity_24h',
 'like_count_velocity_24h',
 'comment_count_velocity_24h',
 'subscriber_count_velocity_24h',
 'view_velocity_upload',
 'like_velocity_upload',
 'view_velocity_per_sub_24h',
 'like_velocity_per_sub_24h',
 'view_velocity_ratio',
 'view_velocity_acceleration',
 'like_velocity_acceleration',
 'view_count_upload_vs_baseline',
 'like_count_upload_vs_baseline',
 'like_rate_upload',
 'like_rate_24h',
 'views_per_sub_upload',
 'likes_per_sub_upload',
 'comments_per_sub_upload',
 'views_per_sub_24h',
 'likes_per_sub_24h',
 'comments_per_sub_24h',
 'title_category',
 'desc_category',
 'title_length',
 'title_word_count',
 'desc_length',
 'desc_link_count',
 'desc_hashta

## Trainer (Grid Search)

With `config_tune.tune_models=True`, `ModelTrainer` runs `RandomizedSearchCV`
on LR, RF, and XGB before fitting the final models on the full train split. Search isn't needed for
Ensemble because it is using search params from RF and XGB.

In [24]:
stages_tune.trainer.run(run_tune)
run_tune.summary()

Loaded hyperparams 'v1.0' (saved 2026-04-21)
  Models: ['LogisticRegression', 'RandomForestClassifier', 'XGBClassifier']
  Search: {'strategy': 'none', 'note': 'pre-tuning baseline'}
Loaded hyperparams from snapshot 'v1.0'.
using param_grid: {'max_iter': [2000, 5000], 'solver': ['saga'], 'penalty': ['elasticnet'], 'l1_ratio': [0.1, 0.5, 0.9], 'C': [0.1, 1.0, 10.0]}
Tuning LogisticRegression with strategy='random' (cv=5, scoring='roc_auc')
  Sampling 100 combinations from grid of ~18 total
Fitting 5 folds for each of 18 candidates, totalling 90 fits


/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 18 is smaller than n_iter=100. Running 18 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning:


Best roc_auc: 0.7569
Best params:   {'solver': 'saga', 'penalty': 'elasticnet', 'max_iter': 5000, 'l1_ratio': 0.9, 'C': 10.0}
using param_grid: {'max_depth': [None, 5, 10, 20, 30], 'min_samples_split': [2, 5, 10], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [2, 5, 10], 'n_estimators': [300, 500, 800]}
Tuning RandomForestClassifier with strategy='random' (cv=5, scoring='roc_auc')
  Sampling 100 combinations from grid of ~270 total
Fitting 5 folds for each of 100 candidates, totalling 500 fits

Best roc_auc: 0.8608
Best params:   {'n_estimators': 800, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}
using param_grid: {'colsample_bytree': [0.6, 0.7, 0.8, 1.0], 'gamma': [0, 0.1, 0.3, 0.5], 'max_depth': [3, 4, 5, 6, 8], 'min_child_weight': [1, 3, 5, 10], 'subsample': [0.6, 0.7, 0.8, 1.0], 'learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2], 'n_estimators': [300, 500, 1000, 1500], 'tree_method': ['hist'], 'max_bin': [256], 'reg_alpha': [0, 0.01,

## HyperparamSnapshotter — Persist Tuned Params

In [25]:
if not dry_run:
    stages_tune.hyperparam_snapshotter.run(run_tune)
else:
    print('[dry_run] skipping HyperparamSnapshotter')

Saved hyperparams 'v1.1' → gs://maduros-dolce-capstone-data/hyperparams/v1.1.json
  Models: ['LogisticRegression', 'RandomForest', 'XGBoost', 'VotingClassifier']
  Search: {'strategy': 'random', 'n_iter': 100, 'cv': 5, 'scoring': 'roc_auc'}
[HyperparamSnapshotter] Saved hyperparams 'v1.1' for ['LogisticRegression', 'RandomForest', 'XGBoost', 'VotingClassifier'].


## Model Snapshotter — Save Tuned Models

In [26]:
if not dry_run:
    stages_tune.model_snapshotter.run(run_tune)
else:
    print('[dry_run] skipping ModelSnapshotter')

Model artifacts saved to models/v5.1_lr_l1/
  Uploaded gs://maduros-dolce-capstone-data/models/v5.1_lr_l1/model.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v5.1_lr_l1/scaler.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v5.1_lr_l1/feature_cols.json
  Uploaded gs://maduros-dolce-capstone-data/models/v5.1_lr_l1/metadata.json

Model v5.1_lr_l1 (LogisticRegression)
  Data: v3.4_100real (10236 real + 0 synthetic)
  ROC-AUC: 0.7638
  Accuracy: 0.7012
  F1 (above): 0.7374
Model artifacts saved to models/v5.1_rf/
  Uploaded gs://maduros-dolce-capstone-data/models/v5.1_rf/model.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v5.1_rf/scaler.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v5.1_rf/feature_cols.json
  Uploaded gs://maduros-dolce-capstone-data/models/v5.1_rf/metadata.json

Model v5.1_rf (RandomForest)
  Data: v3.4_100real (10236 real + 0 synthetic)
  ROC-AUC: 0.8657
  Accuracy: 0.7844
  F1 (above): 0.8111
Model artifacts saved to models/v5.1_xgb/
  

## Validator — Evaluate on Locked Holdout

In [27]:
stages_tune.validator.run(run_tune)
run_tune.summary()


=== Validator — validation-set results ===
  lr_l1           AUC=0.7608  acc=0.6957  F1↑=0.7308
  rf              AUC=0.8719  acc=0.7949  F1↑=0.8221
  xgb             AUC=0.9083  acc=0.8282  F1↑=0.8466
  ensemble        AUC=0.9054  acc=0.8282  F1↑=0.8468
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 1), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models', 'hyperparams', 'tune']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(18403, 40)
  df_engineered  populated  DataFrame shape=(17989, 87)
  df_train       populated  DataFrame shape=(10236, 87)
  df_test        populated  DataFrame shape=(2560, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  X_train        populated  DataFrame shape=(10236, 53)
  X_test         populated  DataFrame shape=(2560, 53)
  X_val          populated  

## Validation Results Snapshotter

In [28]:
if not dry_run:
    stages_tune.validation_results_snapshotter.run(run_tune)
    # Appends to gs://maduros-dolce-capstone-data/models/{model_version}/validation_results.jsonl
else:
    print('[dry_run] skipping ValidationResultsSnapshotter')

Validation results appended → gs://maduros-dolce-capstone-data/models/v5.1/validation_results.jsonl
  lr_l1           AUC=0.7608  acc=0.6957  F1↑=0.7308
  rf              AUC=0.8719  acc=0.7949  F1↑=0.8221
  xgb             AUC=0.9083  acc=0.8282  F1↑=0.8466
  ensemble        AUC=0.9054  acc=0.8282  F1↑=0.8468


## Commit Tuning Config to GCS

In [29]:
if not dry_run:
    config_tune.commit()
    print('Committed to GCS')
else:
    print('[dry_run] skipping config_tune.commit()')

Committed versions.json -> data v3.4, model v5.1, baselines v4.0, hyperparams v1.1
Committed to GCS


Exception ignored in: <function ResourceTracker.__del__ at 0x104905300>
Traceback (most recent call last):
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x107b69300>
Traceback (most recent call last):
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
Chi